In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

LAMBDA = 0.94
DATA_DIR = Path.cwd() / "testfiles_" / "data"
df = pd.read_csv(DATA_DIR / "test2.csv")

# numeric data (listwise)
num = df.select_dtypes(include=[np.number]).dropna()
X = num.to_numpy(float)
n, d = X.shape
if n < 2 or d < 2:
    raise ValueError("Need at least 2 observations and 2 numeric columns.")

# Exponential weights (newest gets largest), finite-sample normalized
w = (1 - LAMBDA) * LAMBDA ** np.arange(n - 1, -1, -1)
w = w / w.sum()

# EW covariance around weighted mean
mu = (w[:, None] * X).sum(axis=0)
Xc = X - mu
S = (w[:, None] * Xc).T @ Xc

# EW correlation
std = np.sqrt(np.diag(S))
eps = 1e-18
std = np.where(std < eps, eps, std)
R = S / np.outer(std, std)

print(pd.DataFrame(R, index=num.columns, columns=num.columns))

          x1        x2        x3        x4        x5
x1  1.000000  0.084787  0.190100  0.129045  0.070706
x2  0.084787  1.000000 -0.077432  0.202228 -0.442546
x3  0.190100 -0.077432  1.000000  0.198267  0.102093
x4  0.129045  0.202228  0.198267  1.000000  0.119541
x5  0.070706 -0.442546  0.102093  0.119541  1.000000
